# AGENTS_030 — Change Impact Agent (Hackathon Demo)

This notebook:
1. Clones **your fork** (`rajipsv/TheRock`, branch `feature/change-impact-agent`)
2. Installs Python dependencies
3. Runs unit tests (`pytest`)
4. Analyzes real upstream PRs on `ROCm/TheRock`
5. Generates executive summaries and displays HTML reports

**Secrets:** paste `GITHUB_TOKEN` in `agents/change-impact-agent/.env` (gitignored) — never hardcode tokens in this notebook.

**Out of scope:** ARVIL integration, cross-repo pattern scanning (see `SCOPE.md` in the agent folder).

In [4]:
import os
import subprocess
import sys
from pathlib import Path

# --- Configuration (edit if needed) ---
FORK_REPO = "https://github.com/rajipsv/TheRock.git"
BRANCH = "feature/change-impact-agent"
CLONE_DIR = Path(os.environ.get("THEROCK_CLONE", Path.home() / "TheRock-fork-demo"))
UPSTREAM_REPO = "ROCm/TheRock"
# PRs used in the hackathon demo (upstream ROCm/TheRock)
DEMO_PRS = [5572, 5688, 5480, 5718]

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
if GITHUB_TOKEN:
    print("GITHUB_TOKEN set — superrepo component detection enabled")
else:
    print("GITHUB_TOKEN not set — some PRs may have partial component lists (rate limits)")

print(f"Clone target: {CLONE_DIR}")

GITHUB_TOKEN set — superrepo component detection enabled
Clone target: C:\Users\Rajeswari\TheRock-fork-demo


## 1. Clone fork (skip if already cloned)

In [5]:
def run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(
            result.returncode, cmd, output=result.stdout, stderr=result.stderr
        )
    return result

if (CLONE_DIR / ".git").exists():
    print(f"Repo exists at {CLONE_DIR} — fetching branch {BRANCH}")
    run(["git", "fetch", "origin", BRANCH], cwd=CLONE_DIR)
    run(["git", "checkout", BRANCH], cwd=CLONE_DIR)
    run(["git", "pull", "origin", BRANCH], cwd=CLONE_DIR, check=False)
    run(["git", "fetch", "origin", BRANCH, "--depth", "80"], cwd=CLONE_DIR, check=False)
    run([
        "git", "fetch",
        f"https://github.com/{UPSTREAM_REPO}.git",
        "main:upstream-main", "--depth", "200",
    ], cwd=CLONE_DIR, check=False)
else:
    CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    run([
        "git", "clone",
        "--branch", BRANCH,
        "--depth", "80",
        FORK_REPO,
        str(CLONE_DIR),
    ])
    run([
        "git", "fetch",
        f"https://github.com/{UPSTREAM_REPO}.git",
        "main:upstream-main", "--depth", "200",
    ], cwd=CLONE_DIR, check=False)

REPO = CLONE_DIR.resolve()
AGENT = REPO / "agents" / "change-impact-agent"
OUT = AGENT / "out"
# Load agents/change-impact-agent/.env if you created it (GITHUB_TOKEN)
sys.path.insert(0, str(AGENT))
from env_loader import load_agent_env
load_agent_env()
if os.environ.get("GITHUB_TOKEN"):
    print("Loaded GITHUB_TOKEN from .env or environment")
print(f"TheRock root: {REPO}")
print(f"Agent: {AGENT}")

$ git clone --branch feature/change-impact-agent --single-branch https://github.com/rajipsv/TheRock.git C:\Users\Rajeswari\TheRock-fork-demo
TheRock root: C:\Users\Rajeswari\TheRock-fork-demo
Agent: C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent


## 2. Install dependencies

In [6]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(AGENT / "requirements.txt")])
run([sys.executable, "-m", "pip", "install", "-q", "pytest"])

$ C:\Users\Rajeswari\AppData\Local\Programs\Python\Python313\python.exe -m pip install -q -r C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\requirements.txt
$ C:\Users\Rajeswari\AppData\Local\Programs\Python\Python313\python.exe -m pip install -q pytest


CompletedProcess(args=['C:\\Users\\Rajeswari\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', '-m', 'pip', 'install', '-q', 'pytest'], returncode=0)

## 3. Run unit tests

In [7]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "agents/change-impact-agent/tests/", "-q"],
    cwd=REPO,
)
if result.returncode != 0:
    raise RuntimeError("pytest failed — fix tests before demo")
print("All unit tests passed.")

All unit tests passed.


## 4. Submodule bump demo (local git range, no PR)

Uses `HEAD~6..HEAD` on the checked-out branch. A shallow fork clone has **no** local `main` ref, so `main~6..main` fails.

In [9]:
demo_out = OUT / "demo-main-range"
run([
    sys.executable, str(AGENT / "analyze.py"),
    "--start", "HEAD~6", "--end", "HEAD",
    "--output-dir", str(demo_out),
], cwd=REPO)
run([
    sys.executable, str(AGENT / "summarize.py"),
    "--backend", "template",
    "--input", str(demo_out / "report.json"),
    "--output", str(demo_out / "executive_summary.md"),
], cwd=REPO)
print((demo_out / "executive_summary.md").read_text(encoding="utf-8"))

$ C:\Users\Rajeswari\AppData\Local\Programs\Python\Python313\python.exe C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\analyze.py --start main~6 --end main --output-dir C:\Users\Rajeswari\TheRock-fork-demo\agents\change-impact-agent\out\demo-main-range


CalledProcessError: Command '['C:\\Users\\Rajeswari\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', 'C:\\Users\\Rajeswari\\TheRock-fork-demo\\agents\\change-impact-agent\\analyze.py', '--start', 'main~6', '--end', 'main', '--output-dir', 'C:\\Users\\Rajeswari\\TheRock-fork-demo\\agents\\change-impact-agent\\out\\demo-main-range']' returned non-zero exit status 1.

## 5. Analyze upstream PRs (ROCm/TheRock)

Fork clones only have `feature/change-impact-agent` — section 1 fetches `upstream-main` from ROCm/TheRock for merge-base. Re-run section 1 if PR analysis fails.

| PR | Type |
|----|------|
| 5572 | MIOpen GHA timeout 60→120 min |
| 5688 | hipDNN CI + artifact TOML |
| 5480 | OpenMPI CMake version file naming |
| 5718 | rocm-libraries superrepo bump |

In [10]:
UPSTREAM_GIT = f"https://github.com/{UPSTREAM_REPO}.git"

def ensure_upstream_main() -> None:
    """Fork single-branch clones have no local main — fetch upstream tip as upstream-main."""
    run([
        "git", "fetch", UPSTREAM_GIT, "main:upstream-main", "--depth", "200",
    ], cwd=REPO, check=False)

def analyze_supports_pr_flag() -> bool:
    help_text = subprocess.run(
        [sys.executable, str(AGENT / "analyze.py"), "--help"],
        cwd=REPO, capture_output=True, text=True,
    )
    return "--pr" in (help_text.stdout + help_text.stderr)

def analyze_upstream_pr(pr: int, pr_out: Path) -> int:
    """Analyze an upstream ROCm/TheRock PR from a fork clone."""
    ensure_upstream_main()
    if analyze_supports_pr_flag():
        return run([
            sys.executable, str(AGENT / "analyze.py"),
            "--pr", str(pr),
            "--full-manifest",
            "--output-dir", str(pr_out),
        ], cwd=REPO, check=False).returncode
    local_ref = f"pr-{pr}"
    run(["git", "fetch", UPSTREAM_GIT, f"pull/{pr}/head:{local_ref}"], cwd=REPO, check=False)
    return run([
        sys.executable, str(AGENT / "analyze.py"),
        "--end", local_ref,
        "--pr-base-ref", "upstream-main",
        "--output-dir", str(pr_out),
    ], cwd=REPO, check=False).returncode

pr_results = {}
for pr in DEMO_PRS:
    pr_out = OUT / f"pr-{pr}"
    print(f"\n{'='*60}\nPR #{pr}\n{'='*60}")
    rc = analyze_upstream_pr(pr, pr_out)
    if rc != 0:
        print(f"Warning: analyze failed for PR #{pr} (exit {rc})")
        continue
    run([
        sys.executable, str(AGENT / "summarize.py"),
        "--backend", "template",
        "--input", str(pr_out / "report.json"),
        "--output", str(pr_out / "executive_summary.md"),
    ], cwd=REPO)
    summary = (pr_out / "executive_summary.md").read_text(encoding="utf-8")
    pr_results[pr] = summary
    # Print first 25 lines of each summary
    print("\n".join(summary.splitlines()[:25]))
    if len(summary.splitlines()) > 25:
        print("...")


PR #5572

PR #5688

PR #5480

PR #5718


## 6. List open upstream PRs (optional, no analyze)

In [ ]:
subprocess.run([
    sys.executable, str(AGENT / "upstream_pr_scan.py"),
    "--max", "5",
], cwd=REPO)

## 7. View HTML report (example: PR #5572)

In [ ]:
from IPython.display import HTML

example_pr = 5572
html_path = OUT / f"pr-{example_pr}" / "report.html"
if html_path.exists():
    HTML(html_path.read_text(encoding="utf-8"))
else:
    print(f"No report at {html_path} — run section 5 first")

## 8. Optional — vLLM executive summary (MI300)

Requires a running OpenAI-compatible server (e.g. vLLM on MI300).

In [ ]:
# Uncomment and adjust base URL / model for your environment
# VLLM_URL = "http://localhost:8000/v1"
# VLLM_MODEL = "meta/llama-3.1-70b-instruct"
# pr_out = OUT / "pr-5572"
# subprocess.run([
#     sys.executable, str(AGENT / "summarize.py"),
#     "--backend", "vllm",
#     "--base-url", VLLM_URL,
#     "--model", VLLM_MODEL,
#     "--input", str(pr_out / "report.json"),
#     "--output", str(pr_out / "executive_summary_vllm.md"),
# ], cwd=REPO, check=True)
# print((pr_out / "executive_summary_vllm.md").read_text())

## 9. Demo checklist

- [ ] `pytest` green
- [ ] PR #5572 → `test:miopen`, `test_filter:quick`
- [ ] PR #5718 → component-scoped `test:*` (needs `GITHUB_TOKEN` for full list)
- [ ] PR #5480 → `test_filter:quick` (third-party packaging)
- [ ] Open `out/pr-*/report.html` in browser
- [ ] Fork Actions: **Change Impact Upstream PR Scan** workflow dispatch